# 4 · Radar rain on dated landslides (dpc-radar + ispra-idrogeo)

rainfall-thresholds imports no other library: it takes a rain series and gives numbers. This
notebook joins it with its two sister libraries, the way Limen does:

- **ispra-idrogeo** gives the landslides with a date, from IdroGEO (ISPRA);
- **dpc-radar** gives the hourly rain at their place, from the radar of the Civil Protection;
- **rainfall-thresholds** cuts the rain into events and compares them with the Italian threshold.

It needs the network, and it is **not** run in CI. The DPC keeps radar frames for about five
months, so only recent landslides can be read.

> **A threshold exceeded is not an alert.** The official alert in Italy is the criticality
> bulletin of the Dipartimento della Protezione Civile. And this is **not a validation**: the
> Italian thresholds are fitted on rain gauges, the radar alone reads differently (in 2026 about
> 20 % more than the gauge-corrected «Merging», and not the same everywhere), and the sample is
> small and uneven.

Data: IdroGEO (ISPRA), CC BY-SA 4.0 · DPC radar, © Dipartimento della Protezione Civile,
CC BY-SA 4.0 · e-ITALICA v4, CC BY 4.0.

In [1]:
import asyncio
from collections import Counter
from datetime import date, timedelta

import numpy as np
from dpc_radar import ABSENT, RadarClient
from ispra_idrogeo import IdroGEO
from ispra_idrogeo.dated import dated_landslides, instant_utc

from rainfall_thresholds import (
    ISTAT_REGIONS,
    calibration_of,
    event_against_threshold,
    macroregion_of,
    threshold_for,
)

SINCE = date(2026, 7, 1)  # the DPC keeps about five months of frames
HOURS = 48  # the rain read before the instant of the landslide
LIMIT = 10  # the most recent landslides: every one costs up to 48 frame downloads

# Movements a rainfall threshold for shallow landslides is about: slides, flows, complex,
# areas of shallow landslides. Falls, sinkholes, lateral spreads and deep-seated
# deformations are left out.
RAINFALL_MOVEMENTS = {2, 4, 5, 7, 11}


def instant(landslide):
    """The instant of a dated landslide, by the convention of the family (ispra-idrogeo's
    `instant_utc`): its time read as Europe/Rome, or with no time the end of its day, the next
    local midnight. Rounded up to the hour: the radar hours are labelled by their end."""
    t = instant_utc(landslide)
    whole = t.replace(minute=0, second=0, microsecond=0)
    return whole if whole == t else whole + timedelta(hours=1)


print(instant_utc({"start": "2010-01-01", "end": "2010-01-01", "time": None}))
print(instant_utc({"start": "2010-07-01", "end": "2010-07-01", "time": None}))

2010-01-01 23:00:00+00:00
2010-07-01 22:00:00+00:00


## 1. Dated landslides, from IdroGEO

In [2]:
async def landslides():
    async with IdroGEO() as idrogeo:
        return await dated_landslides(
            idrogeo, sources=["eventi"], since=SINCE, max_interval_days=1, max_position_m=1000
        )


found = await landslides()
kept = [f for f in found if f.movement in RAINFALL_MOVEMENTS and f.lon is not None]
print(f"{len(found)} landslides since {SINCE}, dated to the day, point within 1 km")
print(f"{len(kept)} of them slides or flows (the rest are falls, sinkholes, n.d.)")
print(Counter(ISTAT_REGIONS[f.region] for f in kept).most_common())

39 landslides since 2026-07-01, dated to the day, point within 1 km
7 of them slides or flows (the rest are falls, sinkholes, n.d.)
[('Lombardia', 3), ('Veneto', 2), ('Trentino-Alto Adige', 1), ('Lazio', 1)]


## 2. The radar rain at each place

For each landslide, the hourly radar rain (SRT1) at its point over the 48 hours before its
**instant**. IdroGEO gives the day, not the hour: by the convention of the family the landslide
takes the end of its day, the next local midnight in Europe/Rome (23:00 UTC in winter, 22:00 in
summer). An hour is labelled by its end, so the last hour read is the one ending at that
midnight. A frame the DPC no longer has, or a point the radar did not measure, is `None`:
**not measured, not dry**.

In [3]:
recent = sorted(kept, key=lambda f: f.start)[-LIMIT:]


def hours_for(landslide):
    end = instant(landslide)
    return [end - timedelta(hours=h) for h in range(HOURS - 1, -1, -1)]


async def series_for(cases):
    wanted = sorted({t for f in cases for t in hours_for(f)})
    frames = {}
    limit = asyncio.Semaphore(8)
    async with RadarClient() as radar:

        async def one(t):
            async with limit:
                frames[t] = await radar.at("SRT1", t)

        await asyncio.gather(*(one(t) for t in wanted))
    out = {}
    for f in cases:
        hours = {}
        for t in hours_for(f):
            frame = frames[t]
            value = None if frame is ABSENT else frame.sample([(f.lon, f.lat)])[0]
            hours[t] = None if value is None or not np.isfinite(value) else float(value)
        out[f.id] = hours
    gone = sum(v is ABSENT for v in frames.values())
    print(f"{len(wanted)} frames asked, {gone} no longer at the DPC")
    return out


series = await series_for(recent)

192 frames asked, 0 no longer at the DPC


## 3. Against the Italian threshold

`macroregion_of` takes the ISTAT region code IdroGEO gives. `event_against_threshold` picks the
strongest event and says whether its rain is a lower bound (`at_least`: hours not measured
inside it or at its edges). `calibration_of` says how the line sits on the region's own
landslides of the past: `high` where more than 5 % of them are below it.

In [4]:
rows = []
for f in recent:
    name = ISTAT_REGIONS[f.region]
    line = threshold_for(macroregion_of(f.region))
    a = event_against_threshold(
        series[f.id],
        line,
        no_rain_break_hours=6,
        min_event_mm=2.0,
        choice="strongest",
        source="DPC SRT1",
    )
    missing = sum(mm is None for mm in series[f.id].values())
    rows.append((f.id, f.start, name, a, missing, calibration_of(f.region).position))

head = ("landslide", "day", "region", "rain", "hours", "exceed.", "not meas.", "line here")
print(
    f"{head[0]:<14}{head[1]:<12}{head[2]:<22}{head[3]:>10}{head[4]:>7}{head[5]:>9}"
    f"{head[6]:>11}  {head[7]}"
)
for id_, day, name, a, missing, position in rows:
    if a is None:
        what = "no rain event" if missing < HOURS else "not measured"
        print(f"{id_:<14}{day!s:<12}{name:<22}{what:>26}{missing:>11}  {position}")
    else:
        e = a.event
        mm = f"{'≥' if a.at_least else ''}{e.total_mm:.1f}mm"
        print(
            f"{id_:<14}{day!s:<12}{name:<22}{mm:>10}{e.duration_hours:>7.0f}"
            f"{a.excess_decades:>+9.2f}{missing:>11}  {position}"
        )

landslide     day         region                      rain  hours  exceed.  not meas.  line here
EIT202600076  2026-08-02  Trentino-Alto Adige       ≥9.5mm      5    -0.17          0  high
EIT202600109  2026-08-02  Veneto                    45.5mm      7    +0.44          0  as fitted
EIT202600078  2026-08-11  Lazio                              no rain event          0  as fitted
EIT202600083  2026-08-20  Lombardia               ≥102.6mm     26    +0.52          0  as fitted
EIT202600086  2026-08-20  Lombardia               ≥153.4mm     28    +0.68          0  as fitted
EIT202600089  2026-08-20  Lombardia                ≥79.4mm     25    +0.41          0  as fitted
EIT202600097  2026-09-10  Veneto                   ≥77.0mm     44    +0.28          0  as fitted


In [5]:
measured = [a for _, _, _, a, _, _ in rows if a is not None]
above = sum(a.ratio > 1 for a in measured)
print(f"{above} of {len(measured)} landslides with a rain event had it above T5 on radar rain")
inside = sum(a.event.unmeasured_hours > 0 for a in measured)
edge = sum(a.borders_unmeasured for a in measured)
print(f"{inside} events have hours not measured inside")
print(f"{edge} touch the edge of the {HOURS} hours read: they may be longer (the ≥ in the table)")
print(f"{len(rows) - len(measured)} with no rain event or no measured rain in the {HOURS} hours")

5 of 6 landslides with a rain event had it above T5 on radar rain
0 events have hours not measured inside
5 touch the edge of the 48 hours read: they may be longer (the ≥ in the table)
1 with no rain event or no measured rain in the 48 hours


## What this says, and what it does not

- It shows the **join**: three libraries, each on its own, put together in a few lines. Limen
  does the same in its scoring and backtests.
- It is **not a measure** of the threshold on radar rain. A handful of recent landslides, mostly in
  the north, are an anecdote: IdroGEO dates to the day less than 1 % of the landslides in 16
  regions. The measure, on all the landslides since 2018 and on the radar history, runs in
  Limen ([#8](https://github.com/agent-engineering-studio/rainfall-thresholds/issues/8)).
- A landslide under the threshold here may be a landslide the radar read low, one triggered by
  the weeks before rather than the day, or one the threshold misses. The notebook cannot tell
  which.
- Nothing here is an alert.